In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import re
import sys
import time
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


path = r'D:\ML\ML_20251\archive'

# Lấy danh sách tất cả file .csv trong thư mục đó
all_files = glob.glob(os.path.join(path, "*.csv"))

print(f"Tìm thấy {len(all_files)} file CSV")

if len(all_files) == 0:
    print(f"Không tìm thấy file CSV nào trong thư mục: {path}")
else:
    # Đọc và gộp toàn bộ các file lại thành 1 DataFrame duy nhất
    df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
    
    print(f"Đã nạp và gộp thành công {len(all_files)} tệp dữ liệu")
    print(f"Tổng số dòng: {len(df)}")
    display(df.head())

In [ ]:
df.to_csv(r'D:\ML\ML_20251\all_products_merged.csv', index=False)
print("Đã lưu file gộp thành công!")

In [ ]:
# Xem thông tin tổng quan
print("=" * 80)
print("DATASET INFORMATION")
print("=" * 80)
df.info()

In [ ]:
# Xem 5 dòng đầu tiên
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Thống kê mô tả
print("\nDescriptive Statistics:")
df.describe()

In [ ]:
# Kiểm tra missing values
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_df.to_string(index=False))

# Visualization
if len(missing_df) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(missing_df['Column'], missing_df['Missing_Percentage'], color='coral')
    plt.xlabel('Missing Percentage (%)')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.show()

##  SƠ ĐỒ TỔNG QUAN MỐI QUAN HỆ CÁC TRƯỜNG QUAN TRỌNG

In [ ]:
print("=" * 80)
print("1. CORRELATION MATRIX - MỐI QUAN HỆ GIỮA CÁC TRƯỜNG SỐ")
print("=" * 80)

df_corr = df.copy()

if 'ratings' in df_corr.columns:
    df_corr['ratings_num'] = pd.to_numeric(df_corr['ratings'], errors='coerce')

if 'no_of_ratings' in df_corr.columns:
    df_corr['no_of_ratings_num'] = df_corr['no_of_ratings'].astype(str).str.replace(',', '')
    df_corr['no_of_ratings_num'] = pd.to_numeric(df_corr['no_of_ratings_num'], errors='coerce')

def extract_price(price_str):
    if pd.isna(price_str):
        return np.nan
    price_str = str(price_str).replace('₹', '').replace(',', '').strip()
    try:
        return float(price_str)
    except:
        return np.nan

if 'discount_price' in df_corr.columns:
    df_corr['discount_price_num'] = df_corr['discount_price'].apply(extract_price)

if 'actual_price' in df_corr.columns:
    df_corr['actual_price_num'] = df_corr['actual_price'].apply(extract_price)

numeric_cols = ['ratings_num', 'no_of_ratings_num', 'discount_price_num', 'actual_price_num']
numeric_cols = [col for col in numeric_cols if col in df_corr.columns]

correlation_matrix = df_corr[numeric_cols].corr()

print("\nCorrelation Matrix:")
print(correlation_matrix.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.3f', square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Mối quan hệ giữa các trường số', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("PHÂN TÍCH CORRELATION QUAN TRỌNG:")
print("=" * 80)

strong_correlations = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = correlation_matrix.iloc[i, j]
        if abs(corr_value) > 0.5:
            strong_correlations.append({
                'Feature 1': correlation_matrix.columns[i],
                'Feature 2': correlation_matrix.columns[j],
                'Correlation': corr_value
            })

if strong_correlations:
    strong_corr_df = pd.DataFrame(strong_correlations)
    print("\nCác mối quan hệ mạnh (|r| > 0.5):")
    print(strong_corr_df.to_string(index=False))
else:
    print("\nKhông có mối quan hệ mạnh (|r| > 0.5)")

print("\nÝ nghĩa:")
print("  • r gần +1: Tương quan thuận mạnh")
print("  • r gần -1: Tương quan nghịch mạnh")
print("  • r gần 0: Không có tương quan")

In [ ]:
print("=" * 80)
print("2. PHAN PHOI CAC TRUONG QUAN TRONG")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 2.1 Distribution of Ratings
if 'ratings_num' in df_corr.columns:
    ax1 = axes[0, 0]
    ratings_clean = df_corr['ratings_num'].dropna()
    ax1.hist(ratings_clean, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    ax1.axvline(ratings_clean.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {ratings_clean.mean():.2f}')
    ax1.axvline(ratings_clean.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {ratings_clean.median():.2f}')
    ax1.set_xlabel('Ratings', fontsize=12)
    ax1.set_ylabel('Tan suat', fontsize=12)
    ax1.set_title('Phan phoi Ratings', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)

# 2.2 Distribution of Number of Ratings (Log scale)
if 'no_of_ratings_num' in df_corr.columns:
    ax2 = axes[0, 1]
    no_ratings_clean = df_corr['no_of_ratings_num'].dropna()
    no_ratings_clean = no_ratings_clean[no_ratings_clean > 0]  # Remove zeros for log scale
    ax2.hist(np.log10(no_ratings_clean + 1), bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Log10(So luong Danh gia + 1)', fontsize=12)
    ax2.set_ylabel('Tan suat', fontsize=12)
    ax2.set_title('Phan phoi So luong Danh gia (Log Scale)', fontsize=14, fontweight='bold')
    ax2.grid(alpha=0.3)

# 2.3 Distribution of Discount Price (Log scale)
if 'discount_price_num' in df_corr.columns:
    ax3 = axes[1, 0]
    price_clean = df_corr['discount_price_num'].dropna()
    price_clean = price_clean[price_clean > 0]
    ax3.hist(np.log10(price_clean), bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
    ax3.set_xlabel('Log10(Gia giam)', fontsize=12)
    ax3.set_ylabel('Tan suat', fontsize=12)
    ax3.set_title('Phan phoi Gia giam (Log Scale)', fontsize=14, fontweight='bold')
    ax3.grid(alpha=0.3)

# 2.4 Price Distribution: Discount vs Actual
if 'discount_price_num' in df_corr.columns and 'actual_price_num' in df_corr.columns:
    ax4 = axes[1, 1]
    discount_clean = df_corr['discount_price_num'].dropna()
    actual_clean = df_corr['actual_price_num'].dropna()
    
    ax4.hist(discount_clean, bins=50, alpha=0.6, label='Gia giam', color='orange', edgecolor='black')
    ax4.hist(actual_clean, bins=50, alpha=0.6, label='Gia goc', color='purple', edgecolor='black')
    ax4.set_xlabel('Gia (₹)', fontsize=12)
    ax4.set_ylabel('Tan suat', fontsize=12)
    ax4.set_title('Gia giam vs Gia goc', fontsize=14, fontweight='bold')
    ax4.legend()
    ax4.grid(alpha=0.3)
    ax4.set_xlim(0, df_corr[['discount_price_num', 'actual_price_num']].quantile(0.95).max())

plt.tight_layout()
plt.show()

# Statistics summary
print("\nTHONG KE MO TA:")
print("=" * 80)
print(df_corr[numeric_cols].describe().round(2))

In [ ]:
print("=" * 80)
print("3. MOI QUAN HE GIUA GIA VA RATING")
print("=" * 80)

# 3.1 Scatter plot: Price vs Ratings
if 'discount_price_num' in df_corr.columns and 'ratings_num' in df_corr.columns:
    plt.figure(figsize=(10, 6))
    
    # Lọc dữ liệu hợp lệ
    valid_data = df_corr[['discount_price_num', 'ratings_num']].dropna()
    valid_data = valid_data[(valid_data['discount_price_num'] > 0) & (valid_data['ratings_num'] > 0)]
    
    # Lấy mẫu để visualization nhanh hơn nếu dữ liệu quá lớn
    if len(valid_data) > 10000:
        valid_data = valid_data.sample(10000, random_state=42)
    
    # Scatter plot
    scatter = plt.scatter(valid_data['discount_price_num'], 
                          valid_data['ratings_num'],
                          alpha=0.3, s=20, c='blue', edgecolors='none')
    
    # Trend line
    z = np.polyfit(valid_data['discount_price_num'], valid_data['ratings_num'], 1)
    p = np.poly1d(z)
    plt.plot(valid_data['discount_price_num'].sort_values(), 
             p(valid_data['discount_price_num'].sort_values()), 
             "r--", linewidth=2, label='Duong xu huong')
    
    plt.xlabel('Gia giam (₹)', fontsize=12)
    plt.ylabel('Ratings', fontsize=12)
    plt.title('Moi quan he: Gia vs Rating', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.xlim(0, valid_data['discount_price_num'].quantile(0.95))
    plt.tight_layout()
    plt.show()

# Tính correlation
if 'discount_price_num' in df_corr.columns and 'ratings_num' in df_corr.columns:
    valid_data = df_corr[['discount_price_num', 'ratings_num']].dropna()
    corr = valid_data['discount_price_num'].corr(valid_data['ratings_num'])
    print(f"\nCorrelation (Gia vs Rating): {corr:.4f}")
    
    if abs(corr) < 0.1:
        print("   ➜ Khong co moi quan he ro rang")
    elif corr > 0:
        print("   ➜ Tuong quan thuan: Gia cao → Rating cao")
    else:
        print("   ➜ Tuong quan nghich: Gia cao → Rating thap")

In [ ]:
# Kiểm tra duplicate records
duplicates = df.duplicated().sum()
print(f"\nDuplicate Records: {duplicates:,}")

# Kiểm tra duplicate theo tên sản phẩm
if 'name' in df.columns:
    name_duplicates = df.duplicated(subset=['name']).sum()
    print(f"Duplicate Product Names: {name_duplicates:,}")

#  BƯỚC 1: XỬ LÝ MISSING VALUES

Xử lý các giá trị thiếu trong dataset

In [ ]:
# Tạo bản sao để xử lý
df_clean = df.copy()

print("=" * 80)
print("XỬ LÝ MISSING VALUES")
print("=" * 80)

# 1. Xóa cột không cần thiết (Unnamed: 0)
if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop('Unnamed: 0', axis=1)
    print(" Đã xóa cột 'Unnamed: 0'")

# 2. Xử lý ratings: chuyển sang numeric trước
if 'ratings' in df_clean.columns:
    df_clean['ratings'] = pd.to_numeric(df_clean['ratings'], errors='coerce')
    if df_clean['ratings'].isnull().sum() > 0:
        median_rating = df_clean['ratings'].median()
        df_clean['ratings'].fillna(median_rating, inplace=True)
        print(f" Đã điền ratings bằng median: {median_rating}")

# 3. no_of_ratings: chuyển về số và xử lý
if 'no_of_ratings' in df_clean.columns:
    # Xóa dấu phẩy và chuyển thành số
    df_clean['no_of_ratings'] = df_clean['no_of_ratings'].astype(str).str.replace(',', '')
    df_clean['no_of_ratings'] = pd.to_numeric(df_clean['no_of_ratings'], errors='coerce')
    df_clean['no_of_ratings'].fillna(0, inplace=True)
    df_clean['no_of_ratings'] = df_clean['no_of_ratings'].astype(int)
    print("Đã xử lý no_of_ratings")

# 4. Xóa các dòng có missing values ở cột quan trọng
important_cols = ['name', 'main_category', 'sub_category']
before_drop = len(df_clean)
df_clean = df_clean.dropna(subset=important_cols)
after_drop = len(df_clean)
print(f"Đã xóa {before_drop - after_drop:,} dòng thiếu thông tin quan trọng")

# 5. Các cột còn lại: điền bằng giá trị mặc định
df_clean['image'].fillna('no_image', inplace=True)
df_clean['link'].fillna('no_link', inplace=True)

print(f"\nKích thước dataset sau xử lý: {df_clean.shape}")
print(f"Missing values còn lại: {df_clean.isnull().sum().sum()}")

#  BƯỚC 2: XỬ LÝ GIÁ CẢ (PRICE CLEANING)

Chuyển đổi cột giá từ string sang numeric

In [ ]:
print("=" * 80)
print("XỬ LÝ GIÁ CẢ")
print("=" * 80)

# Hàm xử lý giá cả
def clean_price(price_str):
    """Chuyển đổi giá từ string sang số"""
    if pd.isna(price_str):
        return np.nan
    # Xóa ký tự ₹ và dấu phẩy
    price_str = str(price_str).replace('₹', '').replace(',', '').strip()
    try:
        return float(price_str)
    except:
        return np.nan

# Áp dụng cho discount_price và actual_price
df_clean['discount_price_num'] = df_clean['discount_price'].apply(clean_price)
df_clean['actual_price_num'] = df_clean['actual_price'].apply(clean_price)

# Xử lý missing values cho giá
# Nếu discount_price bị thiếu, dùng actual_price
df_clean['discount_price_num'].fillna(df_clean['actual_price_num'], inplace=True)
# Nếu actual_price bị thiếu, dùng discount_price
df_clean['actual_price_num'].fillna(df_clean['discount_price_num'], inplace=True)

# Tính % giảm giá
df_clean['discount_percentage'] = ((df_clean['actual_price_num'] - df_clean['discount_price_num']) / 
                                     df_clean['actual_price_num'] * 100).round(2)
df_clean['discount_percentage'].fillna(0, inplace=True)

# Xóa các sản phẩm không có giá
before_drop = len(df_clean)
df_clean = df_clean.dropna(subset=['discount_price_num', 'actual_price_num'])
after_drop = len(df_clean)

print(f"Đã chuyển đổi giá từ string sang numeric")
print(f"Đã tính % giảm giá")
print(f"Đã xóa {before_drop - after_drop:,} sản phẩm không có giá")
print(f"\nGiá trung bình (sau giảm): ₹{df_clean['discount_price_num'].mean():,.2f}")
print(f"Giá trung bình (trước giảm): ₹{df_clean['actual_price_num'].mean():,.2f}")
print(f"% giảm giá trung bình: {df_clean['discount_percentage'].mean():.2f}%")

#  BƯỚC 3: XỬ LÝ DUPLICATES

Xử lý các sản phẩm trùng lặp

In [ ]:
print("=" * 80)
print("XỬ LÝ DUPLICATES")
print("=" * 80)

before_drop = len(df_clean)

# Xóa duplicate hoàn toàn giống nhau
df_clean = df_clean.drop_duplicates()
after_exact_dup = len(df_clean)
print(f"Đã xóa {before_drop - after_exact_dup:,} bản ghi duplicate hoàn toàn")

# Xóa duplicate theo tên sản phẩm (giữ lại sản phẩm có nhiều đánh giá nhất)
df_clean = df_clean.sort_values('no_of_ratings', ascending=False)
df_clean = df_clean.drop_duplicates(subset=['name'], keep='first')
after_name_dup = len(df_clean)
print(f"Đã xóa {after_exact_dup - after_name_dup:,} sản phẩm trùng tên")

print(f"\nTổng số dòng đã xóa: {before_drop - after_name_dup:,}")
print(f"Kích thước dataset sau xử lý: {df_clean.shape}")

#  BƯỚC 4: XỬ LÝ TEXT DATA

Chuẩn hóa dữ liệu text (tên sản phẩm, category)

In [ ]:
print("=" * 80)
print("XỬ LÝ TEXT DATA")
print("=" * 80)

# 1. Chuẩn hóa tên sản phẩm
# Xóa khoảng trắng thừa
df_clean['name'] = df_clean['name'].str.strip()
df_clean['name'] = df_clean['name'].str.replace(r'\s+', ' ', regex=True)

# 2. Chuẩn hóa category (chuyển về lowercase)
df_clean['main_category'] = df_clean['main_category'].str.lower().str.strip()
df_clean['sub_category'] = df_clean['sub_category'].str.strip()

# 3. Tạo cột combined_text cho recommendation system
# Lặp sub_category 3 lần để tăng trọng số, giảm main_category
df_clean['combined_text'] = (
    df_clean['name'] + ' ' + 
    df_clean['sub_category'] + ' ' + 
    df_clean['sub_category'] + ' ' + 
    df_clean['sub_category'] + ' ' + 
    df_clean['main_category']
)

# 4. Xử lý text cho recommendation: lowercase và xóa ký tự đặc biệt
def process_text(text):
    """Xử lý text để chuẩn bị cho recommendation"""
    if pd.isna(text):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # Chỉ giữ chữ, số và khoảng trắng
    text = re.sub(r'\s+', ' ', text)  # Xóa khoảng trắng thừa
    return text.strip()

df_clean['processed_text'] = df_clean['combined_text'].apply(process_text)

print("Đã chuẩn hóa tên sản phẩm")
print("Đã chuẩn hóa category")
print("Đã tạo combined_text và processed_text")

# Hiển thị ví dụ

print("\nVí dụ:")
print(f"Processed text: {df_clean['processed_text'].iloc[0][:80]}...")
print(f"Tên gốc: {df_clean['name'].iloc[0][:80]}...")

#  BƯỚC 5: XỬ LÝ OUTLIERS

Xử lý các giá trị ngoại lai trong giá và ratings

In [ ]:
print("=" * 80)
print("XỬ LÝ OUTLIERS")
print("=" * 80)

before_outlier = len(df_clean)

# 1. Xử lý outliers cho giá (loại bỏ giá âm và giá quá cao/thấp bất thường)
# Loại bỏ giá <= 0
df_clean = df_clean[(df_clean['discount_price_num'] > 0) & (df_clean['actual_price_num'] > 0)]
print(f" Đã loại bỏ giá <= 0")

# Loại bỏ outliers bằng IQR method
Q1_price = df_clean['discount_price_num'].quantile(0.25)
Q3_price = df_clean['discount_price_num'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound = Q1_price - 3 * IQR_price
upper_bound = Q3_price + 3 * IQR_price

df_clean = df_clean[
    (df_clean['discount_price_num'] >= lower_bound) & 
    (df_clean['discount_price_num'] <= upper_bound)
]
print(f"Đã loại bỏ outliers giá (IQR method)")
print(f" Khoảng giá hợp lệ: ₹{lower_bound:,.2f} - ₹{upper_bound:,.2f}")

# 2. Xử lý ratings (phải từ 1-5)
df_clean = df_clean[(df_clean['ratings'] >= 1) & (df_clean['ratings'] <= 5)]
print(f"Đã loại bỏ ratings không hợp lệ")

# 3. Xử lý % giảm giá bất thường (> 100% hoặc < 0%)
df_clean = df_clean[
    (df_clean['discount_percentage'] >= 0) & 
    (df_clean['discount_percentage'] <= 100)
]
print(f" Đã loại bỏ % giảm giá bất thường")

after_outlier = len(df_clean)
print(f"\n Đã xóa {before_outlier - after_outlier:,} outliers")
print(f" Kích thước dataset cuối cùng: {df_clean.shape}")

#  BƯỚC 6: TẠO FEATURES MỚI

Tạo thêm các đặc trưng hữu ích cho phân tích

In [ ]:
print("=" * 80)
print("TẠO FEATURES MỚI")
print("=" * 80)

# 1. Phân loại mức giá
def categorize_price(price):
    if price < 1000:
        return 'Budget'
    elif price < 5000:
        return 'Mid-Range'
    elif price < 20000:
        return 'Premium'
    else:
        return 'Luxury'

df_clean['price_category'] = df_clean['discount_price_num'].apply(categorize_price)
print("Đã tạo price_category")

# 2. Phân loại ratings
def categorize_rating(rating):
    if rating >= 4.5:
        return 'Excellent'
    elif rating >= 4.0:
        return 'Very Good'
    elif rating >= 3.5:
        return 'Good'
    elif rating >= 3.0:
        return 'Average'
    else:
        return 'Below Average'

df_clean['rating_category'] = df_clean['ratings'].apply(categorize_rating)
print("Đã tạo rating_category")

# 3. Phân loại mức độ phổ biến dựa vào số lượng đánh giá
def categorize_popularity(count):
    if count >= 10000:
        return 'Very Popular'
    elif count >= 1000:
        return 'Popular'
    elif count >= 100:
        return 'Moderate'
    elif count > 0:
        return 'Low'
    else:
        return 'No Reviews'

df_clean['popularity'] = df_clean['no_of_ratings'].apply(categorize_popularity)
print(" Đã tạo popularity")

# 4. Tính tiết kiệm được bao nhiêu tiền
df_clean['savings_amount'] = df_clean['actual_price_num'] - df_clean['discount_price_num']
print("Đã tạo savings_amount")

# 5. Tính popularity score (ratings * log(no_of_ratings + 1))
df_clean['popularity_score'] = df_clean['ratings'] * np.log1p(df_clean['no_of_ratings'])
print("Đã tạo popularity_score")

print("\n Features mới đã tạo:")
print("   - price_category")
print("   - rating_category")
print("   - popularity")
print("   - savings_amount")
print("   - popularity_score")

#  BƯỚC 7: TỔNG KẾT & LƯU DỮ LIỆU

Kiểm tra kết quả cuối cùng và lưu dữ liệu đã xử lý

In [ ]:
print("=" * 80)
print("TỔNG KẾT DỮ LIỆU SAU TIỀN XỬ LÝ")
print("=" * 80)

# Thông tin tổng quan
print(f"Kích thước dataset: {df_clean.shape}")
print(f"Số cột: {len(df_clean.columns)}")
print(f"Số sản phẩm: {len(df_clean):,}")
print(f"Số categories: {df_clean['main_category'].nunique()}")
print(f"Số sub-categories: {df_clean['sub_category'].nunique()}")

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)
missing_check = df_clean.isnull().sum()
if missing_check.sum() == 0:
    print("Không còn missing values")
else:
    print(missing_check[missing_check > 0])

print("\n" + "=" * 80)
print("THỐNG KÊ GIÁ")
print("=" * 80)
print(df_clean[['discount_price_num', 'actual_price_num', 'discount_percentage', 'savings_amount']].describe())

print("\n" + "=" * 80)
print("PHÂN PHỐI CATEGORIES")
print("=" * 80)
print(df_clean['price_category'].value_counts())
print("\n")
print(df_clean['rating_category'].value_counts())
print("\n")
print(df_clean['popularity'].value_counts())

print("\n" + "=" * 80)
print("TOP 10 MAIN CATEGORIES")
print("=" * 80)
print(df_clean['main_category'].value_counts().head(10))

In [ ]:
# Lưu dữ liệu đã xử lý
output_file = r'D:\ML\ML_20251\amazon_products_cleaned.csv'
df_clean.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Đã lưu dữ liệu đã xử lý vào: {output_file}")
print(f"Kích thước file: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

In [ ]:
# Hiển thị mẫu dữ liệu cuối cùng
print("\n Mẫu dữ liệu đã xử lý:")
display(df_clean[['name', 'main_category', 'sub_category', 'ratings', 'no_of_ratings', 
                   'discount_price_num', 'discount_percentage', 'price_category', 
                   'rating_category', 'popularity', 'popularity_score']].head(10))

#  VISUALIZATION - PHÂN TÍCH DỮ LIỆU

Trực quan hóa dữ liệu sau khi tiền xử lý

In [ ]:
# Thiết lập style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Tạo figure với multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Phân phối giá
axes[0, 0].hist(df_clean['discount_price_num'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Giá (₹)', fontsize=12)
axes[0, 0].set_ylabel('Số lượng sản phẩm', fontsize=12)
axes[0, 0].set_title('Phân phối Giá Sản phẩm', fontsize=14, fontweight='bold')
axes[0, 0].axvline(df_clean['discount_price_num'].mean(), color='red', linestyle='--', label=f"Mean: ₹{df_clean['discount_price_num'].mean():,.0f}")
axes[0, 0].legend()

# 2. Phân phối ratings
axes[0, 1].hist(df_clean['ratings'], bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Ratings', fontsize=12)
axes[0, 1].set_ylabel('Số lượng sản phẩm', fontsize=12)
axes[0, 1].set_title('Phân phối Ratings', fontsize=14, fontweight='bold')
axes[0, 1].axvline(df_clean['ratings'].mean(), color='red', linestyle='--', label=f"Mean: {df_clean['ratings'].mean():.2f}")
axes[0, 1].legend()

# 3. Price Category
price_cat_counts = df_clean['price_category'].value_counts()
axes[1, 0].bar(price_cat_counts.index, price_cat_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
axes[1, 0].set_xlabel('Price Category', fontsize=12)
axes[1, 0].set_ylabel('Số lượng sản phẩm', fontsize=12)
axes[1, 0].set_title('Phân phối theo Mức giá', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Rating Category
rating_cat_counts = df_clean['rating_category'].value_counts()
axes[1, 1].barh(rating_cat_counts.index, rating_cat_counts.values, color='coral')
axes[1, 1].set_xlabel('Số lượng sản phẩm', fontsize=12)
axes[1, 1].set_ylabel('Rating Category', fontsize=12)
axes[1, 1].set_title('Phân phối theo Đánh giá', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 categories
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 Main Categories
top_main = df_clean['main_category'].value_counts().head(10)
axes[0].barh(top_main.index, top_main.values, color='steelblue')
axes[0].set_xlabel('Số lượng sản phẩm', fontsize=12)
axes[0].set_title('Top 10 Main Categories', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Top 10 Sub Categories
top_sub = df_clean['sub_category'].value_counts().head(10)
axes[1].barh(top_sub.index, top_sub.values, color='mediumseagreen')
axes[1].set_xlabel('Số lượng sản phẩm', fontsize=12)
axes[1].set_title('Top 10 Sub Categories', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Chọn các cột numeric để tính correlation
numeric_cols = ['ratings', 'no_of_ratings', 'discount_price_num', 'actual_price_num', 
                'discount_percentage', 'savings_amount', 'popularity_score']
corr_matrix = df_clean[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Correlation Matrix - Các Đặc trưng Numeric', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

#  HỆ THỐNG GỢI Ý SẢN PHẨM TƯƠNG TỰ
##  So sánh 3 thuật toán: Cosine Similarity, kNN, và Matrix Factorization

Xây dựng và so sánh hiệu suất của 3 thuật toán recommendation

In [ ]:
# Import thêm thư viện cho recommendation system
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

print("Đã import các thư viện cho Recommendation System!")

In [ ]:
#Lấy sample để demo (có thể dùng toàn bộ dataset nếu muốn)
df_rec = df_clean.sample(n=min(10000, len(df_clean)), random_state=42).reset_index(drop=True)
print(f" Sử dụng {len(df_rec):,} sản phẩm cho recommendation system")

# Vectorize text data với TF-IDF
print("\n Đang vectorize text data...")
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8
)

# Tạo TF-IDF matrix từ processed_text
tfidf_matrix = tfidf.fit_transform(df_rec['processed_text'])
print(f" TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f" Số features: {tfidf_matrix.shape[1]}")

##  Bước 1: Chuẩn bị dữ liệu cho Recommendation System

Lấy sample và vectorization text data

---
##  THUẬT TOÁN 1: COSINE SIMILARITY

Gợi ý dựa trên độ tương đồng nội dung

In [ ]:
print("=" * 80)
print("THUẬT TOÁN 1: COSINE SIMILARITY (On-Demand)")
print("=" * 80)

# Không precompute toàn bộ matrix - sẽ tính on-demand để tiết kiệm memory
start_time = time.time()
cosine_time = time.time() - start_time

print(f"Sử dụng On-Demand Cosine Similarity")
print(f"Similarity sẽ được tính khi cần thiết (tiết kiệm memory)")
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Dataset size: {len(df_rec):,} sản phẩm")

# Hàm gợi ý sản phẩm sử dụng Cosine Similarity (On-Demand)
def recommend_cosine(product_name, top_n=10):
    """
    Gợi ý sản phẩm dựa trên Cosine Similarity (tính on-demand)
    
    Parameters:
    - product_name: Tên hoặc một phần tên sản phẩm
    - top_n: Số lượng sản phẩm gợi ý
    
    Returns:
    - DataFrame chứa sản phẩm được gợi ý
    """
    try:
        # Tìm sản phẩm
        idx = df_rec[df_rec['name'].str.contains(product_name, case=False, na=False)].index[0]
        product_info = df_rec.iloc[idx]
        
        # Tính similarity chỉ cho sản phẩm này với toàn bộ dataset
        product_vector = tfidf_matrix[idx]
        text_similarities = cosine_similarity(product_vector, tfidf_matrix).flatten()
        
        # Tính price similarity (based on price difference)
        product_price = df_rec.iloc[idx]['discount_price_num']
        price_diffs = np.abs(df_rec['discount_price_num'] - product_price)
        max_price_diff = price_diffs.max()
        price_similarities = 1 - (price_diffs / (max_price_diff + 1))  # Normalize 0-1
        
        # Tính ratings similarity
        product_rating = df_rec.iloc[idx]['ratings']
        rating_diffs = np.abs(df_rec['ratings'] - product_rating)
        ratings_similarities = 1 - (rating_diffs / 4.0)  # Max diff = 4 (5-1)
        
        # Tính popularity similarity (based on popularity_score)
        product_popularity = df_rec.iloc[idx]['popularity_score']
        popularity_diffs = np.abs(df_rec['popularity_score'] - product_popularity)
        max_popularity_diff = popularity_diffs.max()
        popularity_similarities = 1 - (popularity_diffs / (max_popularity_diff + 1))
        
        # Kết hợp: 50% text + 25% price + 15% ratings + 10% popularity
        combined_similarities = (0.50 * text_similarities + 
                                0.25 * price_similarities + 
                                0.15 * ratings_similarities + 
                                0.10 * popularity_similarities)
        
        # Lấy top N indices (bỏ chính nó)
        sim_scores = list(enumerate(combined_similarities))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:top_n+1]  # Bỏ chính nó
        
        # Lấy indices và scores
        product_indices = [i[0] for i in sim_scores]
        similarity_scores = [i[1] for i in sim_scores]
        
        # Tạo DataFrame kết quả
        result = df_rec.iloc[product_indices][['name', 'main_category', 'sub_category', 
                                                 'ratings', 'discount_price_num']].copy()
        result['similarity_score'] = [f"{score:.4f}" for score in similarity_scores]
        result['method'] = 'Cosine Similarity'
        
        print(f"\n Sản phẩm gốc: {product_info['name']}")
        print(f" Category: {product_info['main_category']}")
        print(f" Giá: ₹{product_info['discount_price_num']:,.0f}")
        print(f" Rating: {product_info['ratings']}\n")
        
        return result
        
    except:
        print("Không tìm thấy sản phẩm!")
        return pd.DataFrame()

print("\nHàm recommend_cosine đã sẵn sàng!")

---
##  THUẬT TOÁN 2: k-NEAREST NEIGHBORS (kNN)

Tìm k sản phẩm gần nhất dựa trên khoảng cách

In [ ]:
print("=" * 80)
print("THUẬT TOÁN 2: k-NEAREST NEIGHBORS (kNN)")
print("=" * 80)

# Train kNN model
start_time = time.time()
knn_model = NearestNeighbors(
    n_neighbors=21,  # Lấy 21 để bỏ chính nó còn 20
    metric='cosine',
    algorithm='brute',  # Tốt cho sparse matrix
    n_jobs=-1
)
knn_model.fit(tfidf_matrix)
knn_time = time.time() - start_time

print(f" Đã train kNN model")
print(f"  Thời gian training: {knn_time:.2f} giây")
print(f" Metric: cosine distance")
print(f" K neighbors: 20")

# Hàm gợi ý sản phẩm sử dụng kNN
def recommend_knn(product_name, top_n=10):
    """
    Gợi ý sản phẩm dựa trên k-Nearest Neighbors
    
    Parameters:
    - product_name: Tên hoặc một phần tên sản phẩm
    - top_n: Số lượng sản phẩm gợi ý
    
    Returns:
    - DataFrame chứa sản phẩm được gợi ý
    """
    try:
        # Tìm sản phẩm
        idx = df_rec[df_rec['name'].str.contains(product_name, case=False, na=False)].index[0]
        product_info = df_rec.iloc[idx]
        
        # Lấy vector của sản phẩm
        product_vector = tfidf_matrix[idx]
        product_price = df_rec.iloc[idx]['discount_price_num']
        
        # Tìm nhiều neighbors hơn để filter theo giá
        distances, indices = knn_model.kneighbors(product_vector, n_neighbors=min(top_n*3, len(df_rec)))
        
        # Bỏ sản phẩm đầu tiên (chính nó)
        indices = indices.flatten()[1:]
        distances = distances.flatten()[1:]
        
        # Tính text similarity
        text_similarities = 1 - distances
        
        # Tính price similarity cho candidates
        price_diffs = np.abs(df_rec.iloc[indices]['discount_price_num'].values - product_price)
        max_price_diff = price_diffs.max() if price_diffs.max() > 0 else 1
        price_similarities = 1 - (price_diffs / (max_price_diff + 1))
        
        # Tính ratings similarity
        product_rating = df_rec.iloc[idx]['ratings']
        rating_diffs = np.abs(df_rec.iloc[indices]['ratings'].values - product_rating)
        ratings_similarities = 1 - (rating_diffs / 4.0)
        
        # Tính popularity similarity
        product_popularity = df_rec.iloc[idx]['popularity_score']
        popularity_diffs = np.abs(df_rec.iloc[indices]['popularity_score'].values - product_popularity)
        max_popularity_diff = popularity_diffs.max() if popularity_diffs.max() > 0 else 1
        popularity_similarities = 1 - (popularity_diffs / (max_popularity_diff + 1))
        
        # Kết hợp: 50% text + 25% price + 15% ratings + 10% popularity
        combined_scores = (0.50 * text_similarities + 
                          0.25 * price_similarities + 
                          0.15 * ratings_similarities + 
                          0.10 * popularity_similarities)
        
        # Lấy top N
        top_indices_idx = np.argsort(combined_scores)[::-1][:top_n]
        indices = indices[top_indices_idx]
        similarity_scores = combined_scores[top_indices_idx]
        
        # Tạo DataFrame kết quả
        result = df_rec.iloc[indices][['name', 'main_category', 'sub_category', 
                                         'ratings', 'discount_price_num']].copy()
        result['similarity_score'] = [f"{score:.4f}" for score in similarity_scores]
        result['method'] = 'kNN'
        
        print(f"\n Sản phẩm gốc: {product_info['name']}")
        print(f" Category: {product_info['main_category']}")
        print(f" Giá: ₹{product_info['discount_price_num']:,.0f}")
        print(f" Rating: {product_info['ratings']}\n")
        
        return result
        
    except:
        print("Không tìm thấy sản phẩm!")
        return pd.DataFrame()

print("\nHàm recommend_knn đã sẵn sàng!")

---
##  THUẬT TOÁN 3: MATRIX FACTORIZATION

Sử dụng SVD (Singular Value Decomposition) để giảm chiều dữ liệu

In [ ]:
print("=" * 80)
print("THUẬT TOÁN 3: MATRIX FACTORIZATION (SVD)")
print("=" * 80)

# Sử dụng TruncatedSVD để giảm chiều
n_components = 100  # Số chiều giảm xuống

start_time = time.time()
svd_model = TruncatedSVD(n_components=n_components, random_state=42)
svd_features = svd_model.fit_transform(tfidf_matrix)
svd_time = time.time() - start_time

print(f"Đã train SVD model")
print(f"Thời gian training: {svd_time:.2f} giây")
print(f"Original dimensions: {tfidf_matrix.shape[1]}")
print(f"Reduced dimensions: {n_components}")
print(f"Explained variance: {svd_model.explained_variance_ratio_.sum():.2%}")

# Tính similarity matrix từ reduced features
svd_sim_matrix = cosine_similarity(svd_features, svd_features)

# Hàm gợi ý sản phẩm sử dụng Matrix Factorization
def recommend_svd(product_name, top_n=10):
    """
    Gợi ý sản phẩm dựa trên Matrix Factorization (SVD)
    
    Parameters:
    - product_name: Tên hoặc một phần tên sản phẩm
    - top_n: Số lượng sản phẩm gợi ý
    
    Returns:
    - DataFrame chứa sản phẩm được gợi ý
    """
    try:
        # Tìm sản phẩm
        idx = df_rec[df_rec['name'].str.contains(product_name, case=False, na=False)].index[0]
        product_info = df_rec.iloc[idx]
        
        # Lấy text similarity từ SVD
        text_similarities = svd_sim_matrix[idx]
        
        # Tính price similarity
        product_price = df_rec.iloc[idx]['discount_price_num']
        price_diffs = np.abs(df_rec['discount_price_num'] - product_price)
        max_price_diff = price_diffs.max()
        price_similarities = 1 - (price_diffs / (max_price_diff + 1))
        
        # Tính ratings similarity
        product_rating = df_rec.iloc[idx]['ratings']
        rating_diffs = np.abs(df_rec['ratings'] - product_rating)
        ratings_similarities = 1 - (rating_diffs / 4.0)  # Max diff = 4
        
        # Tính popularity similarity
        product_popularity = df_rec.iloc[idx]['popularity_score']
        popularity_diffs = np.abs(df_rec['popularity_score'] - product_popularity)
        max_popularity_diff = popularity_diffs.max()
        popularity_similarities = 1 - (popularity_diffs / (max_popularity_diff + 1))
        
        # Kết hợp: 50% text + 25% price + 15% ratings + 10% popularity
        combined_similarities = (0.50 * text_similarities + 
                                0.25 * price_similarities + 
                                0.15 * ratings_similarities + 
                                0.10 * popularity_similarities)
        
        # Lấy top N
        sim_scores = list(enumerate(combined_similarities))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:top_n+1]  # Bỏ chính nó
        
        # Lấy indices và scores
        product_indices = [i[0] for i in sim_scores]
        similarity_scores = [i[1] for i in sim_scores]
        
        # Tạo DataFrame kết quả
        result = df_rec.iloc[product_indices][['name', 'main_category', 'sub_category', 
                                                 'ratings', 'discount_price_num']].copy()
        result['similarity_score'] = [f"{score:.4f}" for score in similarity_scores]
        result['method'] = 'SVD (Matrix Factorization)'
        
        print(f"\n Sản phẩm gốc: {product_info['name']}")
        print(f" Category: {product_info['main_category']}")
        print(f" Giá: ₹{product_info['discount_price_num']:,.0f}")
        print(f" Rating: {product_info['ratings']}\n")
        
        return result
        
    except:
        print("Không tìm thấy sản phẩm!")
        return pd.DataFrame()

print("\nHàm recommend_svd đã sẵn sàng!")

---
##  TEST VÀ SO SÁNH CÁC THUẬT TOÁN

Chạy test và so sánh hiệu suất của 3 thuật toán

In [ ]:
# Chọn một sản phẩm để test
from IPython.display import HTML, display

test_product = df_rec.iloc[100]['name']
test_image = df_rec.iloc[100]['image']

print("=" * 80)
print("TEST VỚI SẢN PHẨM MẪU")
print("=" * 80)
print(f" Sản phẩm test: {test_product}")
print(f" Category: {df_rec.iloc[100]['main_category']}")
print(f" Sub-category: {df_rec.iloc[100]['sub_category']}")
print(f" Rating: {df_rec.iloc[100]['ratings']}")
print(f" Giá: ₹{df_rec.iloc[100]['discount_price_num']:,.0f}")
print("=" * 80)

# Hiển thị hình ảnh sản phẩm test
html = f'''
<div style="text-align: center; margin: 20px; padding: 20px; border: 2px solid #4ECDC4; border-radius: 10px; background-color: #f9f9f9;">
    <h3 style="color: #4ECDC4;"> SẢN PHẨM TEST</h3>
    <img src="{test_image}" style="width: 300px; height: 300px; object-fit: cover; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);" 
         onerror="this.src='https://via.placeholder.com/300?text=No+Image'"/>
    <div style="margin-top: 15px; font-size: 14px; line-height: 1.6;">
        <strong style="font-size: 16px; color: #333;">{test_product}</strong><br/>
        <span style="color: #666;">Category: {df_rec.iloc[100]['main_category']} | Sub: {df_rec.iloc[100]['sub_category']}</span><br/>
        <span style="color: #FF6B6B; font-weight: bold;"> {df_rec.iloc[100]['ratings']}</span> | 
        <span style="color: #4ECDC4; font-weight: bold;"> ₹{df_rec.iloc[100]['discount_price_num']:,.0f}</span>
    </div>
</div>
'''
display(HTML(html))

In [ ]:
# So sánh hiệu suất
print("\n" + "=" * 80)
print(" SO SÁNH HIỆU SUẤT CÁC THUẬT TOÁN")
print("=" * 80)

comparison_df = pd.DataFrame({
    'Thuật toán': ['Cosine Similarity (On-Demand)', 'kNN', 'SVD (Matrix Factorization)'],
    'Training Time (s)': [cosine_time, knn_time, svd_time],
    'Query Time (ms)': [time_cosine_query*1000, time_knn_query*1000, time_svd_query*1000],
    'Memory (MB)': [
        tfidf_matrix.data.nbytes / (1024**2),  # Chỉ lưu TF-IDF matrix
        tfidf_matrix.data.nbytes / (1024**2),
        svd_features.nbytes / (1024**2)
    ]
})

print(comparison_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training time
axes[0].bar(comparison_df['Thuật toán'], comparison_df['Training Time (s)'], 
            color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0].set_ylabel('Thời gian (giây)', fontsize=12)
axes[0].set_title('Thời gian Training', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(comparison_df['Training Time (s)']):
    axes[0].text(i, v, f'{v:.2f}s', ha='center', va='bottom')

# Query time
axes[1].bar(comparison_df['Thuật toán'], comparison_df['Query Time (ms)'], 
            color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_ylabel('Thời gian (ms)', fontsize=12)
axes[1].set_title('Thời gian Query', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(comparison_df['Query Time (ms)']):
    axes[1].text(i, v, f'{v:.2f}ms', ha='center', va='bottom')

# Memory usage
axes[2].bar(comparison_df['Thuật toán'], comparison_df['Memory (MB)'], 
            color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[2].set_ylabel('Dung lượng (MB)', fontsize=12)
axes[2].set_title('Memory Usage', fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=15)
for i, v in enumerate(comparison_df['Memory (MB)']):
    axes[2].text(i, v, f'{v:.1f}MB', ha='center', va='bottom')

plt.tight_layout()
plt.show()

##  HIỂN THỊ SẢN PHẨM KÈM HÌNH ẢNH

Hàm hiển thị trực quan sản phẩm gợi ý với hình ảnh

###  Demo: Test hàm hiển thị với hình ảnh

Thử nghiệm các hàm hiển thị sản phẩm với hình ảnh

In [ ]:
# Test Cosine Similarity với hiển thị hình ảnh
from IPython.display import HTML, display

print("THUẬT TOÁN 1: COSINE SIMILARITY")
start = time.time()
rec_cosine = recommend_cosine(test_product[:20], top_n=5)
time_cosine_query = time.time() - start
print(f"Thời gian query: {time_cosine_query*1000:.2f}ms\n")

# Hiển thị với hình ảnh dạng HTML grid
if not rec_cosine.empty:
    html = '<div style="display: flex; flex-wrap: wrap; gap: 20px;">'
    
    for idx, (df_idx, product) in enumerate(rec_cosine.iterrows()):
        # Lấy image từ df_rec sử dụng index
        image_url = df_rec.loc[df_idx, 'image']
        
        html += f'''
        <div style="border: 1px solid #ddd; padding: 10px; width: 250px; text-align: center;">
            <img src="{image_url}" style="width: 200px; height: 200px; object-fit: cover;" 
                 onerror="this.src='https://via.placeholder.com/200?text=No+Image'"/>
            <div style="margin-top: 10px; font-size: 12px;">
                <strong>{product['name'][:40]}...</strong><br/>
                 {product['ratings']} |  ₹{product['discount_price_num']:,.0f}<br/>
                Similarity: {product['similarity_score']}
            </div>
        </div>
        '''
    
    html += '</div>'
    display(HTML(html))
    
    # Hiển thị bảng chi tiết
    print("\n CHI TIẾT SẢN PHẨM GỢI Ý:")
    display(rec_cosine)

In [ ]:
# Ví dụ 2: Test kNN với hiển thị hình ảnh
from IPython.display import HTML, display

print("THUẬT TOÁN 2: k-NEAREST NEIGHBORS")
start = time.time()
rec_knn = recommend_knn(test_product[:20], top_n=5)
time_knn_query = time.time() - start
print(f"Thời gian query: {time_knn_query*1000:.2f}ms\n")

# Hiển thị với hình ảnh dạng HTML grid
if not rec_knn.empty:
    html = '<div style="display: flex; flex-wrap: wrap; gap: 20px;">'
    
    for idx, (df_idx, product) in enumerate(rec_knn.iterrows()):
        # Lấy image từ df_rec sử dụng index
        image_url = df_rec.loc[df_idx, 'image']
        
        html += f'''
        <div style="border: 1px solid #ddd; padding: 10px; width: 250px; text-align: center;">
            <img src="{image_url}" style="width: 200px; height: 200px; object-fit: cover;" 
                 onerror="this.src='https://via.placeholder.com/200?text=No+Image'"/>
            <div style="margin-top: 10px; font-size: 12px;">
                <strong>{product['name'][:40]}...</strong><br/>
                 {product['ratings']} |  ₹{product['discount_price_num']:,.0f}<br/>
                Similarity: {product['similarity_score']}
            </div>
        </div>
        '''
    
    html += '</div>'
    display(HTML(html))
    
    # Hiển thị bảng chi tiết
    print("\n CHI TIẾT SẢN PHẨM GỢI Ý:")
    display(rec_knn)

In [ ]:
# Ví dụ 3: Test SVD với hiển thị hình ảnh
from IPython.display import HTML, display

print("THUẬT TOÁN 3: MATRIX FACTORIZATION (SVD)")
start = time.time()
rec_svd = recommend_svd(test_product[:20], top_n=5)
time_svd_query = time.time() - start
print(f"Thời gian query: {time_svd_query*1000:.2f}ms\n")

# Hiển thị với hình ảnh dạng HTML grid
if not rec_svd.empty:
    html = '<div style="display: flex; flex-wrap: wrap; gap: 20px;">'
    
    for idx, (df_idx, product) in enumerate(rec_svd.iterrows()):
        # Lấy image từ df_rec sử dụng index
        image_url = df_rec.loc[df_idx, 'image']
        
        html += f'''
        <div style="border: 1px solid #ddd; padding: 10px; width: 250px; text-align: center;">
            <img src="{image_url}" style="width: 200px; height: 200px; object-fit: cover;" 
                 onerror="this.src='https://via.placeholder.com/200?text=No+Image'"/>
            <div style="margin-top: 10px; font-size: 12px;">
                <strong>{product['name'][:40]}...</strong><br/>
                 {product['ratings']} |  ₹{product['discount_price_num']:,.0f}<br/>
                Similarity: {product['similarity_score']}
            </div>
        </div>
        '''
    
    html += '</div>'
    display(HTML(html))
    
    # Hiển thị bảng chi tiết
    print("\nCHI TIẾT SẢN PHẨM GỢI Ý:")
    display(rec_svd)